<h1>Chapter 4 - Preparing Data for Vector Stores</h1>
<i>Prepare and chunk text data to store them in a vector store.</i>

<a href="https://learning.oreilly.com/library/view/rag-with-python/9798341600553/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/polzerdo55862/RAG-with-Python-Cookbook"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/polzerdo55862/RAG-with-Python-Cookbook/blob/main/ch04_data_preparation_chunking_data/chunking_data.ipynb)

---

This notebook is for Chapter 4 of the [RAG with Python Cookbook](https://learning.oreilly.com/library/view/rag-with-python/9798341600553/) book by [Dominik Polzer](https://www.linkedin.com/in/polzerdo/).

---

<a href="https://learning.oreilly.com/library/view/rag-with-python/9798341600553/">
  <img src="https://raw.githubusercontent.com/polzerdo55862/RAG-with-Python-Cookbook/main/rag_cookbook.png" width="350" />
</a>


## Prerequisits

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to uncomment and run the following codeblock to install the dependencies for this chapter.

In [1]:
!pip install PyPDF2==3.0.1
!pip install pandas==2.2.3
!pip install pydantic==2.11.5
!pip install openai==1.83.0
!pip install matplotlib==3.10.3
!pip install scikit-learn==1.6.1
!pip install python-docx==1.1.2
!pip install nltk==3.9.1
!pip install langchain==0.3.25
# !pip install langchain_openai==0.3.21
!pip install langchain-experimental==0.3.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 79.6 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.2.3 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.2/67.2 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.2/444.2 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 29.1 MB/s eta 0:00:00
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.41.4
    Uninstall

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.3/244.3 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 450.8/450.8 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.0/363.0 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 61.5 MB/s eta 0:00:00
  Attempting uninstall: zstandard
    Found existing installation: zstandard 0.25.0
    Uninstalling zstandard-0.25.0:
      Successfully uninstalled zstandard-0.25.0
  Attempting uninstall: langsmith
    Found existing installation: langsmith 0.4.59
    Uninstalling langsmith-0.4.59:
      Successfully uninstalled langsmith-0.4.59
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.1
    Uninstalling langchain-core-1.2.1:
      Successfully uninstalled langchain-core-1.2.1
  Attempting uninstall: langchain
    Found existing installation: langchain 1.2.0
    

### Load sample files

This notebook uses sample Word and PDF files.

When running the notebook on Google Colab, uncomment the code below to download the `datasets` directory from the Github repo.

In [3]:
!git clone --no-checkout https://github.com/polzerdo55862/RAG-with-Python-Cookbook.git
%cd RAG-with-Python-Cookbook
!git sparse-checkout init --cone
!git sparse-checkout set datasets
!git checkout
!cp -r datasets /content/datasets


fatal: destination path 'RAG-with-Python-Cookbook' already exists and is not an empty directory.
/content/RAG-with-Python-Cookbook
Your branch is up to date with 'origin/main'.


### Load secrets

If you run this code in Google Colab, save your OpenAI API key in the secrets and access it by

In [4]:
from google.colab import userdata
import os

api_key = userdata.get("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY not found in Colab Secrets")

os.environ["OPENAI_API_KEY"] = api_key

1. Adding Metadata to Enable Metadata Filtering

In [5]:
# tag::load_pdf_and_metadata[]
import PyPDF2
import os

file_path = "../datasets/pdf_files/attention_is_all_you_need_paper.pdf"

with open(file_path, "rb") as file:
    reader = PyPDF2.PdfReader(file)
    metadata = reader.metadata

    text = ""
    for page in reader.pages:
        text += page.extract_text()
# end::load_pdf_and_metadata[]

In [6]:
metadata

{'/Author': '',
 '/CreationDate': 'D:20240410211143Z',
 '/Creator': 'LaTeX with hyperref',
 '/Keywords': '',
 '/ModDate': 'D:20240410211143Z',
 '/PTEX.Fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
 '/Producer': 'pdfTeX-1.40.25',
 '/Subject': '',
 '/Title': '',
 '/Trapped': '/False'}

In [7]:
# tag::generate_customized_metadata[]
metadata_ext = dict(metadata)
metadata_ext["page_count"] = len(reader.pages)
metadata_ext["file_size"] = os.path.getsize(file_path)
metadata_ext["file_name"] = os.path.basename(file_path)
metadata_ext["file_path"] = file_path
metadata_ext["text_length"] = len(text)
# end::generate_customized_metadata[]

In [8]:
metadata_ext

{'/Author': '',
 '/CreationDate': 'D:20240410211143Z',
 '/Creator': 'LaTeX with hyperref',
 '/Keywords': '',
 '/ModDate': 'D:20240410211143Z',
 '/PTEX.Fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
 '/Producer': 'pdfTeX-1.40.25',
 '/Subject': '',
 '/Title': '',
 '/Trapped': '/False',
 'page_count': 15,
 'file_size': 2215244,
 'file_name': 'attention_is_all_you_need_paper.pdf',
 'file_path': '../datasets/pdf_files/attention_is_all_you_need_paper.pdf',
 'text_length': 39472}

In [9]:
# tag::extract_metadata_from_text_using_LLMs[]
from pydantic import BaseModel
from openai import OpenAI

client = OpenAI()

class AuthorContact(BaseModel):
    name: str
    company: str
    email: list[str]

class Contacts(BaseModel):
    entries: list[AuthorContact]

system_message = """Extract the contact information of all authors."""

response = client.responses.parse(
    model="gpt-4.1-mini",
    input=[
        {
            "role": "system",
            "content": system_message,
        },
        {
            "role": "user",
            "content": text,
        },
    ],
    text_format=Contacts,
)

author_contacts = response.output_parsed

metadata_ext["author_contacts"] = author_contacts
# end::extract_metadata_from_text_using_LLMs[]



In [10]:
metadata

{'/Author': '',
 '/CreationDate': 'D:20240410211143Z',
 '/Creator': 'LaTeX with hyperref',
 '/Keywords': '',
 '/ModDate': 'D:20240410211143Z',
 '/PTEX.Fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
 '/Producer': 'pdfTeX-1.40.25',
 '/Subject': '',
 '/Title': '',
 '/Trapped': '/False'}

### 2.2 Enhancing Data Quality by Replacing Abbreviations and Technical Terms

In [12]:
# tag::load_file_and_replace_abbreviations[]
import re

abbreviations_dict = {
    "NLP": "Natural Language Processing",
    "RNN": "Recurrent Neural Network",
    "LSTM": "Long Short-Term Memory",
    "GRU": "Gated Recurrent Unit",
    "TF": "Transformer",
    "MHA": "Multi-Head Attention",
    "FFN": "Feed-Forward Network",
}

file_path = "../datasets/text_files/blog_post_transformers.txt"
with open(file_path, "r") as file:
    text = file.read()

# Replace abbreviations in the text
for abbr, full_form in abbreviations_dict.items():
    text = re.sub(rf"\b{abbr}\b", f"{full_form} ({abbr})", text)
# end::load_file_and_replace_abbreviations[]


In [ ]:
text

In [13]:
# tag::make_text_chunks_self_explanatory[]
import os
from openai import OpenAI

file_path = "../datasets/text_files/EMEA_drives_revenue.txt"

with open(file_path, "r") as file:
    text = file.read()

prompt = f"""
    The text below contains a financial report including a lot of abbreviations and
    technical terms from the finance domain. Please replace the abbreviations with
    their full forms and provide a brief explanation of the technical terms, so the
    whole text get's easier to read and understandable for everyone.

    Make sure it's easy enough, that a 10 years old school kid could understand it.

    Often used abbreviations are:
    - EMEA: Europe, Middle East, and Africa
    - BD: Business Development
    - YoY: Year-over-Year
    - APAC: Asia-Pacific

    Text:
    {text}
    """.strip()

client = OpenAI()
chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": prompt,
        }
    ],
    model="gpt-5",
)

enhanced_text = chat_completion.choices[0].message.content

# end::make_text_chunks_self_explanatory[]

In [14]:
enhanced_text

'Here is the same report with all abbreviations written out and the tricky terms explained in simple words.\n\nEurope, Middle East, and Africa drives sales jump in April–June 2019: Business Development team shows strong Year-over-Year growth\n\nIn April–June 2019 (the second quarter of 2019), the Business Development team helped the company sell much more than before. The Europe, Middle East, and Africa region was the main reason for this jump in sales. Compared to the same time last year, sales were about 2.3 times bigger (a little more than double).\n\nKey highlights\n- Europe, Middle East, and Africa leads: Sales across 20 countries in Europe, the Middle East, and Africa powered the strong April–June 2019 results.\n- Top helpers (share of total sales):\n  - United States: 28%\n  - Spain: 20%\n  - India: 15%\n  - Norway: 11%\n- Year-over-Year growth: The Business Development team grew sales a lot compared to the same months last year.\n\nBreakdown by region\n- Europe, Middle East, an

In [15]:
# write enhanced_text to a new .txt file
output_file_path = "../datasets/text_files/EMEA_drives_revenue_enhanced.txt"
with open(output_file_path, "w") as file:
    file.write(enhanced_text)

### 2.3 Improving Search Accuracy by Embedding Hypothetical Questions

In [ ]:
import PyPDF2

file_path = "../datasets/pdf_files/AI_in_Factories_Discussion_Cleaned.pdf"

with open(file_path, "rb") as file:
    # Create PDF reader object
    reader = PyPDF2.PdfReader(file)

    # Extract text from all pages
    text = ""
    for page in reader.pages:
        text += page.extract_text()


In [17]:
text

"Discussion: Pros and Cons of AI in Factories\nAlex: Hey Sam, I read this article about AI revolutionizing factory work. It sounds amazing! Did you \nknow that AI-powered predictive maintenance can reduce downtime by up to 30%? Imagine how\nmuch that could improve productivity.\nSam: Yeah, I've read about that too. AI definitely has potential, but I feel like there's more to \nconsider than just efficiency. For instance, one study found that automation and AI could displace \nup to 20 million manufacturing jobs worldwide by 2030.\nAlex: That's a fair point, but couldn't companies invest in reskilling programs? According to a report \nby the World Economic Forum, 50% of all employees will need reskilling by 2025 due to the\nadoption of \nnew technologies. It's not impossible-it's about prioritizing it.\nSam: True, but not every company has the resources to reskill workers effectively. And there's also \nthe question of cost. AI implementation is expensive-customized AI solutions for fac

In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    separators=[
        "\n\n",
    ],
)

text_chunks = text_splitter.split_text(text)

In [19]:
# tag::generate_hypothetical_questions[]
import textwrap
from openai import OpenAI
from pydantic import BaseModel

file_path = "../datasets/text_files/AI_in_factories_chat.txt"

with open(file_path, "r", encoding="utf-8") as file:
    text = file.read()

client = OpenAI()

prompt = textwrap.dedent(
    f"""
    Below you can find a chat history between two students.

    Please generate 5 hypothetical questions that could be
    answered using the information from the discussion.
    The questions should focus on key details, definitions, and
    information present in the text.

    Chat History:
    {text}
    """
)

class HypotheticalQuestions(BaseModel):
    questions: list[str]

result = client.responses.parse(
    model="gpt-4o",
    input=prompt,
    text_format=HypotheticalQuestions,
)

hypothetical_questions = result.output_parsed.questions
hypothetical_questions
# end::generate_hypothetical_questions[]


['What percentage can AI systems like robotic process automation reduce manufacturing costs by?',
 'How many workplace injuries occurred in the U.S. in 2022, and what industry were many of them in?',
 'What level of accuracy can machine vision systems achieve in quality control inspections?',
 'What percentage of companies using AI in manufacturing faced system reliability challenges according to a 2023 MIT report?',
 'By what percentage have cyberattacks on industrial systems increased over the past two years?']

In [20]:
hypothetical_questions

['What percentage can AI systems like robotic process automation reduce manufacturing costs by?',
 'How many workplace injuries occurred in the U.S. in 2022, and what industry were many of them in?',
 'What level of accuracy can machine vision systems achieve in quality control inspections?',
 'What percentage of companies using AI in manufacturing faced system reliability challenges according to a 2023 MIT report?',
 'By what percentage have cyberattacks on industrial systems increased over the past two years?']

### 2.4 Splitting Documents Using Character Splitting

In [21]:
# tag::character_text_splitting[]
from langchain.text_splitter import CharacterTextSplitter

file_path = "../datasets/text_files/blog_post_transformers.txt"

# Load example document
with open(file_path, "r") as file:
    text = file.read()

text_splitter = CharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=0,
    separator="",
    length_function=len,
)

text_chunks = text_splitter.create_documents([text])
# end::character_text_splitting[]

In [22]:
text_chunks

[Document(metadata={}, page_content='Unveiling Transformer Models: A Revolution in ML\n\nIn the realm of ML and DL, few advancements have r'),
 Document(metadata={}, page_content='eshaped the field as profoundly as Transformer models. First introduced in the seminal paper "Attent'),
 Document(metadata={}, page_content='ion Is All You Need," Transformer models have become the backbone of modern NLP and are extending th'),
 Document(metadata={}, page_content='eir reach into CV and beyond.\n\nThe Challenges Before Transformers\n\nPrior to Transformers, models lik'),
 Document(metadata={}, page_content='e RNNs, LSTMs, and GRUs were the primary tools for sequential data. While effective, these architect'),
 Document(metadata={}, page_content='ures faced significant challenges:\n\nSequential Computation: Processing one step at a time limited th'),
 Document(metadata={}, page_content='eir ability to leverage parallelism, making training slower.\n\nLong-Range Dependencies: Understanding'),
 Do

### 2.5 Splitting Documents Using Recursive Text Splitters

In [23]:

# tag::recursive_chunking[]
from langchain_text_splitters import RecursiveCharacterTextSplitter
import PyPDF2

file_path = "../datasets/pdf_files/daily_insights.pdf"

with open(file_path, "rb") as file:
    reader = PyPDF2.PdfReader(file)

    text = ""
    for page in reader.pages:
        text += page.extract_text()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=0,
    length_function=len,
    is_separator_regex=False,
)

chunks = text_splitter.split_text(text)
# end::recursive_chunking[]


In [24]:
chunks

['The\nDaily\nInsight\nPolitics:\nNew\nClimate\nAccord\nSets\nAmbitious\nGoals\nWorld\nleaders\nconvened\nthis\nweek\nin\nGeneva\nto\nsign\na\ngroundbreaking\nclimate\naccord\naimed\nat\nreducing\nglobal\ngreenhouse\ngas\nemissions',
 'by\n50%\nby\n2035.\nThis\nlandmark\nagreement,\nsupported\nby\nover\n190\ncountries,\nincludes\nprovisions\nfor\nrenewable\nenergy\ninvestments,\nreforestation\nprojects,\nand\ntechnological\ninnovations.\nCritics,',
 'however,\nargue\nthat\nthe\naccord\nlacks\nbinding\nenforcement\nmechanisms,\nwhich\ncould\nundermine\nits\nimpact.\nSports:\nThrilling\nChampionship\nFinal\nThe\nglobal\nspotlight\nwas\non\nthe\nNational\nFootball',
 'Championship\nlast\nnight\nas\nthe\nBlue\nFalcons\ntriumphed\nover\nthe\nSilver\nWolves\nin\na\nnail-biting\nmatch.\nThe\nfinal\nscore,\n28-27,\nwas\ndecided\nby\na\nlast-minute\nfield\ngoal.\nQuarterback\nAlex\nHart\nof\nthe\nBlue',
 'Falcons\nwas\nnamed\nMVP\nfor\nhis\nexceptional\nperformance,\nthrowing\nthree\ntouchdown\

### 2.6 Document Aware Splitting

In [25]:

# tag::document_aware_text_splitter[]
import os

file_path = "../datasets/markdown_files/random_md_code.md"
file_extension = os.path.splitext(file_path)[1]

with open(file_path, "r") as file:
    file_text = file.read()

from langchain_text_splitters import (
    PythonCodeTextSplitter,
    LatexTextSplitter,
    MarkdownHeaderTextSplitter,
)

if file_extension == ".py":
    splitter = PythonCodeTextSplitter(chunk_size=500, chunk_overlap=50)
elif file_extension == ".tex":
    splitter = LatexTextSplitter(chunk_size=500, chunk_overlap=50)
elif file_extension == ".md":
    headers_to_split_on = [
        ("#", "Header 1"),
        ("##", "Header 2"),
        ("###", "Header 3"),
    ]

    splitter = MarkdownHeaderTextSplitter(headers_to_split_on)

chunks = splitter.split_text(file_text)
# end::document_aware_text_splitter[]

In [26]:
chunks

[Document(metadata={'Header 1': 'Random Markdown Page'}, page_content='---'),
 Document(metadata={'Header 1': 'Random Markdown Page', 'Header 2': 'Table of Contents'}, page_content='1. [Introduction](#introduction)\n2. [Code Snippets](#code-snippets)\n3. [Lists](#lists)\n4. [Quotes and Tips](#quotes-and-tips)\n5. [Tables](#tables)\n6. [Images and Links](#images-and-links)  \n---'),
 Document(metadata={'Header 1': 'Random Markdown Page', 'Header 2': 'Introduction'}, page_content='Welcome to a randomly generated Markdown page. This is an example to showcase various Markdown elements.'),
 Document(metadata={'Header 1': 'Random Markdown Page', 'Header 2': 'Introduction', 'Header 3': 'Highlights'}, page_content='- **Simple formatting**: *italics*, **bold**, `inline code`\n- `Monospace text`\n- ~~Strikethrough~~  \n---'),
 Document(metadata={'Header 1': 'Random Markdown Page', 'Header 2': 'Code Snippets'}, page_content='```python\n# Python example\ndef greet(name):\nreturn f"Hello, {name}!"\

### 2.7 Splitting the Text Using Semantic Aware Chunkers

In [27]:
from docx import Document
from openai import OpenAI

file_path = "../datasets/text_files/random-text-about-5-different-stories.txt"

# read the text from the file
with open(file_path, "r") as file:
    text = file.read()

In [ ]:
# tag::langchain_semantic_text_splitting[]
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai.embeddings import OpenAIEmbeddings

text_splitter = SemanticChunker(
    OpenAIEmbeddings(),
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=90,
)

chunks = text_splitter.split_text(text)
# end::langchain_semantic_text_splitting[]

In [28]:
# chunks

In [29]:
# import pandas as pd

# # initialize the api key
# client = OpenAI()
# embedding_model = "text-embedding-3-small"

# import os

# # Initialize the OpenAI client with your API key
# client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# # create new data frame using text chunks list
# embeddings_df = pd.DataFrame(text_chunks).rename(columns={0: "text_chunk"})

# # helper function to get the embeddings for a text chunk
# def _get_embeddings(text_chunk, client, embedding_model):
#     embedding = (
#         client.embeddings.create(input=[text_chunk], model=embedding_model)
#         .data[0]
#         .embedding
#     )

#     return embedding

# # iterate over embeddings_df["text_chunks"] and create a new data frame with the embeddings
# embeddings_df["embeddings"] = embeddings_df["text_chunk"].apply(
#     _get_embeddings, client=client, embedding_model="text-embedding-3-small"
# )

# # split the embeddings column into individual columns for each vector dimension
# embeddings_df = embeddings_df["embeddings"].apply(pd.Series)
# embeddings_df["text_chunk"] = text_chunks

# embeddings_df = from_text_to_embeddings(
#     chunks, client, embedding_model
# )

### 2.8 Splitting Text Using Agentic Chunkers

In [30]:
# tag::agentic_chunking_create_propositions[]
from langchain import hub
from langchain_openai import ChatOpenAI  # Import ChatOpenAI
from pydantic import BaseModel
from typing import List

# pull the prompt template from the langchain hub
obj = hub.pull("wfh/proposal-indexing")

llm = ChatOpenAI(model="gpt-4o")

class Sentences(BaseModel):
    sentences: List[str]

extraction_llm = llm.with_structured_output(Sentences)

# Create the sentence extraction chain
extraction_chain = obj | extraction_llm

propositions = extraction_chain.invoke(
    """
    On July 20, 1969, astronaut Neil Armstrong walked on the moon .
    He was leading the NASA's Apollo 11 mission.
    Armstrong famously said, "That's one small step for man, one
    giant leap for mankind" as he stepped onto the lunar surface.
    """
)

print(propositions)
# end::agentic_chunking_create_propositions[]

/usr/local/lib/python3.12/dist-packages/langsmith/client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


sentences=['On July 20, 1969, astronaut Neil Armstrong walked on the moon.', "Neil Armstrong was leading NASA's Apollo 11 mission.", 'Neil Armstrong famously said, "That\'s one small step for man, one giant leap for mankind."', 'Neil Armstrong said this famous phrase as he stepped onto the lunar surface.']


In [31]:
for sentence in propositions.sentences:
    print(sentence)

On July 20, 1969, astronaut Neil Armstrong walked on the moon.
Neil Armstrong was leading NASA's Apollo 11 mission.
Neil Armstrong famously said, "That's one small step for man, one giant leap for mankind."
Neil Armstrong said this famous phrase as he stepped onto the lunar surface.
